# svr for AM-I

In [ ]:
import os
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.model_selection import KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from src.ml import (
    plot_learning_curve_from_estimator,
    plot_scatter_and_residuals as reusable_plot_scatter_and_residuals,
)

# ========== 配置 ==========
DATA_FOLDER = '../../data/train_test_split'
OUTPUT_FOLDER = '../../results/ml/2-svr-models'
MODEL_SAVE_FOLDER = os.path.join(OUTPUT_FOLDER, 'AM-I-svr-model')
SEED = 42

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

os.makedirs(MODEL_SAVE_FOLDER, exist_ok=True)

# 全局评估结果
all_eval_results = []

# iPhone配色（清新风格）
IPHONE_COLORS = {
    "scatter": "#007AFF",
    "line": "#AEAEB2",
    "text": "#000000"
}

def load_and_prepare_data(train_file, test_file):
    train_df = pd.read_csv(train_file).dropna(subset=ALL_FEATURES + [TARGET_COL])
    test_df = pd.read_csv(test_file).dropna(subset=ALL_FEATURES + [TARGET_COL])

    X_train = train_df[ALL_FEATURES].values
    y_train = train_df[TARGET_COL].values
    X_test = test_df[ALL_FEATURES].values
    y_test = test_df[TARGET_COL].values

    scaler = StandardScaler()
    X_train[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train[:, :len(FEATURE_COLS)])
    X_test[:, :len(FEATURE_COLS)] = scaler.transform(X_test[:, :len(FEATURE_COLS)])

    return X_train, y_train, X_test, y_test, scaler

def objective(trial, X, y):
    C = trial.suggest_float('C', 1e-2, 1e3, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1e1, log=True)
    epsilon = trial.suggest_float('epsilon', 1e-3, 1.0, log=True)

    model = SVR(C=C, gamma=gamma, epsilon=epsilon)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []

    for train_idx, val_idx in kf.split(X):
        X_train_fold, X_val_fold = X[train_idx].copy(), X[val_idx].copy()
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train_fold[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train_fold[:, :len(FEATURE_COLS)])
        X_val_fold[:, :len(FEATURE_COLS)] = scaler.transform(X_val_fold[:, :len(FEATURE_COLS)])

        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        scores.append(r2_score(y_val_fold, y_pred))

    return np.mean(scores)

def plot_learning_curve(estimator, X, y, title, save_path):
    plot_learning_curve_from_estimator(
        estimator=estimator,
        X=X,
        y=y,
        save_path=save_path,
        title=title,
        scoring='r2',
        n_jobs=1,
        train_sizes=np.linspace(0.1, 1.0, 5),
    )

def plot_scatter_and_residuals(y_true, y_pred, base_name):
    reusable_plot_scatter_and_residuals(
        y_true=y_true,
        y_pred=y_pred,
        save_folder=MODEL_SAVE_FOLDER,
        base_name=base_name,
        colors=IPHONE_COLORS,
    )

def train_and_evaluate(train_csv, test_csv):
    base_name = os.path.splitext(os.path.basename(train_csv))[0].replace("_train", "")
    print(f"\n🚀 Training on dataset: {base_name}")

    X_train, y_train, X_test, y_test, scaler = load_and_prepare_data(train_csv, test_csv)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30)

    best_params = study.best_params
    model = SVR(**best_params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    print(f"📊 R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

    joblib.dump(model, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_svr_model.joblib"))
    joblib.dump(scaler, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_scaler.joblib"))

    pd.DataFrame({'y_true': y_test, 'y_pred': y_pred}).to_csv(
        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_predictions.csv"), index=False
    )

    plot_learning_curve(SVR(**best_params), X_train, y_train,
                        f"Learning Curve - {base_name}",
                        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_learning_curve.png"))

    plot_scatter_and_residuals(y_test, y_pred, base_name)

    all_eval_results.append({
        "Dataset": base_name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "C": best_params['C'],
        "gamma": best_params['gamma'],
        "epsilon": best_params['epsilon']
    })

    # Save summary
    summary_df = pd.DataFrame(all_eval_results)
    summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, "AM-I-svr_model_evaluation_summary.csv"), index=False)
    print(f"\n✅ All model evaluation results saved to: AM-I-svr_model_evaluation_summary.csv")

if __name__ == "__main__":
    train_csv = os.path.join(DATA_FOLDER, "AM-I-filtered_with_labels_k4_train.csv")
    test_csv = os.path.join(DATA_FOLDER, "AM-I-filtered_with_labels_k4_test.csv")
    train_and_evaluate(train_csv, test_csv)


🚀 Training on dataset: AM-I-filtered_with_labels_k4


[I 2026-02-28 12:05:30,519] A new study created in memory with name: no-name-602872f8-3b88-4062-8bef-5dfdb5e4b7c9
[I 2026-02-28 12:07:29,537] Trial 0 finished with value: -0.00391148349401158 and parameters: {'C': 0.7459343285726545, 'gamma': 5.669849511478847, 'epsilon': 0.15702970884055384}. Best is trial 0 with value: -0.00391148349401158.
[I 2026-02-28 12:09:25,634] Trial 1 finished with value: 0.6163221890149961 and parameters: {'C': 9.846738873614559, 'gamma': 0.0006026889128682511, 'epsilon': 0.0029375384576328283}. Best is trial 1 with value: 0.6163221890149961.
[I 2026-02-28 12:11:26,176] Trial 2 finished with value: -0.007691250983001652 and parameters: {'C': 0.0195172246414495, 'gamma': 2.1423021757741068, 'epsilon': 0.06358358856676251}. Best is trial 1 with value: 0.6163221890149961.
[I 2026-02-28 12:13:15,242] Trial 3 finished with value: 0.5821680878271069 and parameters: {'C': 34.70266988650411, 'gamma': 0.00012674255898937226, 'epsilon': 0.8123245085588685}. Best is tr

📊 R2: 0.9058 | RMSE: 4.1324 | MAE: 2.9669

✅ All model evaluation results saved to: AM-I-svr_model_evaluation_summary.csv


# SVR FORM AM-II

In [3]:
import os
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.svm import SVR
from sklearn.model_selection import KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# ========== 配置 ==========
DATA_FOLDER = '../../data/train_test_split'
OUTPUT_FOLDER = '../../results/ml/2-svr-models'
MODEL_SAVE_FOLDER = os.path.join(OUTPUT_FOLDER, 'AM-II-svr-model')
SEED = 42

FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

os.makedirs(MODEL_SAVE_FOLDER, exist_ok=True)

# 全局评估结果
all_eval_results = []

# iPhone配色（清新风格）
IPHONE_COLORS = {
    "scatter": "#007AFF",
    "line": "#AEAEB2",
    "text": "#000000"
}

def load_and_prepare_data(train_file, test_file):
    train_df = pd.read_csv(train_file).dropna(subset=ALL_FEATURES + [TARGET_COL])
    test_df = pd.read_csv(test_file).dropna(subset=ALL_FEATURES + [TARGET_COL])

    X_train = train_df[ALL_FEATURES].values
    y_train = train_df[TARGET_COL].values
    X_test = test_df[ALL_FEATURES].values
    y_test = test_df[TARGET_COL].values

    scaler = StandardScaler()
    X_train[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train[:, :len(FEATURE_COLS)])
    X_test[:, :len(FEATURE_COLS)] = scaler.transform(X_test[:, :len(FEATURE_COLS)])

    return X_train, y_train, X_test, y_test, scaler

def objective(trial, X, y):
    C = trial.suggest_float('C', 1e-2, 1e3, log=True)
    gamma = trial.suggest_float('gamma', 1e-4, 1e1, log=True)
    epsilon = trial.suggest_float('epsilon', 1e-3, 1.0, log=True)

    model = SVR(C=C, gamma=gamma, epsilon=epsilon)
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    scores = []

    for train_idx, val_idx in kf.split(X):
        X_train_fold, X_val_fold = X[train_idx].copy(), X[val_idx].copy()
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]

        scaler = StandardScaler()
        X_train_fold[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train_fold[:, :len(FEATURE_COLS)])
        X_val_fold[:, :len(FEATURE_COLS)] = scaler.transform(X_val_fold[:, :len(FEATURE_COLS)])

        model.fit(X_train_fold, y_train_fold)
        y_pred = model.predict(X_val_fold)
        scores.append(r2_score(y_val_fold, y_pred))

    return np.mean(scores)

def plot_learning_curve(estimator, X, y, title, save_path):
    train_sizes, train_scores, valid_scores = learning_curve(
        estimator, X, y, cv=5, scoring='r2', train_sizes=np.linspace(0.1, 1.0, 5), random_state=SEED)

    train_scores_mean = np.mean(train_scores, axis=1)
    valid_scores_mean = np.mean(valid_scores, axis=1)

    plt.figure()
    plt.plot(train_sizes, train_scores_mean, label='Training score')
    plt.plot(train_sizes, valid_scores_mean, label='Validation score')
    plt.xlabel("Training Set Size")
    plt.ylabel("R2 Score")
    plt.title(title)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def iphone_style_ax(ax):
    """Apply iPhone-style aesthetics to matplotlib axes."""
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(2)
    ax.grid(False)

def plot_scatter_and_residuals(y_true, y_pred, base_name):
    # 预测图
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    
    # 应用iPhone样式
    iphone_style_ax(ax)
    ax.set_aspect('equal', adjustable='box')
    
    # 散点图
    plt.scatter(
        y_true, y_pred,
        alpha=0.8,
        s=70,
        color=IPHONE_COLORS['scatter'],
        edgecolors='none'
    )
    
    # 对角线
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims,
             linestyle='--',
             color=IPHONE_COLORS['line'],
             linewidth=3)
    
    # 计算指标
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    # 坐标轴标签
    plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')
    
    # 添加指标文本
    plt.text(
        0.05, 0.95,
        f"R² = {r2:.3f}\nMAE = {mae:.2f}",
        transform=ax.transAxes,
        va='top',
        fontsize=16,
        color=IPHONE_COLORS['text']
    )
    
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_scatter.png"), dpi=600)
    plt.close()

    # 残差图（保持原样）
    residuals = y_pred - y_true
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    ax.tick_params(axis='both', direction='out', length=6, width=1.2)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    plt.grid(False)

    plt.scatter(y_pred, residuals, alpha=0.6, color=IPHONE_COLORS['scatter'])
    plt.axhline(y=0, linestyle='--', color=IPHONE_COLORS['line'], linewidth=2)

    r2_res = r2_score(y_true, y_pred)
    mae_res = mean_absolute_error(y_true, y_pred)

    plt.xlabel("Predicted Retention Time (s)")
    plt.ylabel("Residuals (Predicted - True)")
    plt.title("")
    plt.text(0.5, -0.15, "Residual Plot", ha='center', va='center', transform=ax.transAxes, fontsize=12, color=IPHONE_COLORS['text'])
    plt.text(0.05, 0.95, f"R² = {r2_res:.3f}\nMAE = {mae_res:.3f}", transform=ax.transAxes, verticalalignment='top', fontsize=10, color=IPHONE_COLORS['text'])
    plt.tight_layout()
    plt.savefig(os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_residuals.png"))
    plt.close()

def train_and_evaluate(train_csv, test_csv):
    base_name = os.path.splitext(os.path.basename(train_csv))[0].replace("_train", "")
    print(f"\n🚀 Training on dataset: {base_name}")

    X_train, y_train, X_test, y_test, scaler = load_and_prepare_data(train_csv, test_csv)

    study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=30)

    best_params = study.best_params
    model = SVR(**best_params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    print(f"📊 R2: {r2:.4f} | RMSE: {rmse:.4f} | MAE: {mae:.4f}")

    joblib.dump(model, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_svr_model.joblib"))
    joblib.dump(scaler, os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_scaler.joblib"))

    pd.DataFrame({'y_true': y_test, 'y_pred': y_pred}).to_csv(
        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_predictions.csv"), index=False
    )

    plot_learning_curve(SVR(**best_params), X_train, y_train,
                        f"Learning Curve - {base_name}",
                        os.path.join(MODEL_SAVE_FOLDER, f"{base_name}_learning_curve.png"))

    plot_scatter_and_residuals(y_test, y_pred, base_name)

    all_eval_results.append({
        "Dataset": base_name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "C": best_params['C'],
        "gamma": best_params['gamma'],
        "epsilon": best_params['epsilon']
    })

    # Save summary
    summary_df = pd.DataFrame(all_eval_results)
    summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, "AM-II-svr_model_evaluation_summary.csv"), index=False)
    print(f"\n✅ All model evaluation results saved to: AM-II-svr_model_evaluation_summary.csv")

if __name__ == "__main__":
    train_csv = os.path.join(DATA_FOLDER, "AM-II-filtered_with_labels_k4_train.csv")
    test_csv = os.path.join(DATA_FOLDER, "AM-II-filtered_with_labels_k4_test.csv")
    train_and_evaluate(train_csv, test_csv)


🚀 Training on dataset: AM-II-filtered_with_labels_k4


[I 2026-02-28 13:16:59,347] A new study created in memory with name: no-name-4306fac8-8a8e-4b01-beba-4ef4f1597147
[I 2026-02-28 13:17:05,656] Trial 0 finished with value: -0.02844375598422717 and parameters: {'C': 0.7459343285726545, 'gamma': 5.669849511478847, 'epsilon': 0.15702970884055384}. Best is trial 0 with value: -0.02844375598422717.
[I 2026-02-28 13:17:12,081] Trial 1 finished with value: 0.5243917861907079 and parameters: {'C': 9.846738873614559, 'gamma': 0.0006026889128682511, 'epsilon': 0.0029375384576328283}. Best is trial 1 with value: 0.5243917861907079.
[I 2026-02-28 13:17:18,514] Trial 2 finished with value: -0.027841477043935338 and parameters: {'C': 0.0195172246414495, 'gamma': 2.1423021757741068, 'epsilon': 0.06358358856676251}. Best is trial 1 with value: 0.5243917861907079.
[I 2026-02-28 13:17:23,974] Trial 3 finished with value: 0.478068788173953 and parameters: {'C': 34.70266988650411, 'gamma': 0.00012674255898937226, 'epsilon': 0.8123245085588685}. Best is tri

📊 R2: 0.9082 | RMSE: 2.6960 | MAE: 1.9215

✅ All model evaluation results saved to: AM-II-svr_model_evaluation_summary.csv


# SVR FOR AM-III, AM-IV, AM-V, AM-VI

In [4]:
import os
import glob
import joblib
import optuna
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.svm import SVR
from sklearn.model_selection import KFold, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings

warnings.filterwarnings("ignore")

# ----------------- 配置 -----------------
DATA_FOLDER = '../../data/processed'                 # 数据文件夹（CSV）
MODEL_SAVE_FOLDER = './2-svr-model-other4'
SEED = 42
np.random.seed(SEED)

# 保持和原来一致的特征列
FEATURE_COLS = ['MolWt', 'logP', 'TPSA', 'H_bond_donors', 'H_bond_acceptors']
FP_COLS = [f'col{i}' for i in range(823)]
MG_COLS = [f'fp_{i}' for i in range(1024)]
ALL_FEATURES = FEATURE_COLS + FP_COLS + MG_COLS
TARGET_COL = 'UV_RT-s'

# 限定只处理的文件名前缀 - 只处理三个数据集
ALLOWED_PREFIXES = {
    "AM-III-filtered",
    "AM-IV-filtered",
    "AM-V-filtered",
    "AM-VI-filtered"
}

# 使用 26 核
N_JOBS = 26

os.makedirs(MODEL_SAVE_FOLDER, exist_ok=True)

# iPhone Style Color Palette (匹配参考代码)
IPHONE_COLORS = {
    'scatter': '#007AFF',  # iPhone blue
    'line': '#AEAEB2',     # iPhone gray
    'text': '#000000',     # Black
    'residual': '#34C759'  # 保留残差图颜色
}

# ----------------- 数据加载 -----------------
def load_data():
    data_files = glob.glob(os.path.join(DATA_FOLDER, '*.csv'))
    dfs = []
    for f in data_files:
        file_prefix = os.path.splitext(os.path.basename(f))[0]
        if file_prefix not in ALLOWED_PREFIXES:
            continue
        df = pd.read_csv(f)
        needed_cols = set(ALL_FEATURES + [TARGET_COL])
        if not needed_cols.issubset(set(df.columns)):
            print(f"Warning: file {f} missing required columns, skipping.")
            continue
        df = df.dropna(subset=ALL_FEATURES + [TARGET_COL]).copy()
        if df.shape[0] == 0:
            print(f"Warning: file {f} has no valid rows after dropna, skipping.")
            continue
        df['file_prefix'] = file_prefix
        dfs.append(df)
    if len(dfs) == 0:
        raise RuntimeError("No valid data files found for the allowed prefixes.")
    data = pd.concat(dfs, ignore_index=True)
    return data

# ----------------- plotting helpers -----------------
def iphone_style_ax(ax):
    """Apply iPhone-style aesthetics to matplotlib axes."""
    ax.tick_params(axis='both', direction='out', length=6, width=2, labelsize=16)
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
    ax.grid(False)

def plot_learning_curve(estimator, X, y, title, save_path):
    train_sizes, train_scores, valid_scores = learning_curve(
        estimator, X, y, cv=5, scoring='r2',
        train_sizes=np.linspace(0.1, 1.0, 5), random_state=SEED, n_jobs=N_JOBS)

    train_scores_mean = np.mean(train_scores, axis=1)
    valid_scores_mean = np.mean(valid_scores, axis=1)

    plt.figure()
    plt.plot(train_sizes, train_scores_mean, label='Training score')
    plt.plot(train_sizes, valid_scores_mean, label='Validation score')
    plt.xlabel("Training Set Size")
    plt.ylabel("R2 Score")
    plt.title(title)
    plt.legend(loc='best')
    plt.tight_layout()
    plt.savefig(save_path, dpi=600)
    plt.close()

def plot_scatter_and_residuals(y_true, y_pred, summary, save_prefix):
    """绘制散点图，显示Outer CV的平均值±标准差"""
    
    # 散点图 - 按照参考代码的格式
    plt.figure(figsize=(6, 6))  # Canvas size: 6x6 inches
    ax = plt.gca()
    

    # 在 plot_scatter_and_residuals 函数的散点图部分添加：
    ax.set_aspect('equal', adjustable='box')
    plt.scatter(y_true, y_pred, alpha=0.8, s=70, color=IPHONE_COLORS['scatter'], edgecolors='none')

    
    # Apply iPhone-style axis settings
    iphone_style_ax(ax)
    # 在 iphone_style_ax 函数中添加：
    for spine in ['top', 'right', 'bottom', 'left']:
      ax.spines[spine].set_visible(True)
      ax.spines[spine].set_linewidth(2)  # 添加这行
    
    # Scatter plot specifications
    plt.scatter(
        y_true, y_pred,            # x-axis: true values, y-axis: predicted values
        alpha=0.8,                 # Transparency: 80%
        s=70,                      # Point size: 70
        color=IPHONE_COLORS['scatter']  # Color: iPhone blue (#007AFF)
    )
    
    # Ideal fit line (diagonal)
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    plt.plot(lims, lims,           # Plot y=x diagonal
             linestyle='--',       # Dashed line style
             color=IPHONE_COLORS['line'],  # Color: iPhone gray (#AEAEB2)
             linewidth=3)          # Line width: 3
    
    # 从summary中获取Outer CV的指标
    r2_mean = summary['r2_mean']
    r2_std = summary['r2_std']
    mae_mean = summary['mae_mean']
    mae_std = summary['mae_std']
    
    # 计算当前数据的指标用于显示在图上
    r2_current = r2_score(y_true, y_pred) if len(y_true) > 0 else np.nan
    mae_current = mean_absolute_error(y_true, y_pred) if len(y_true) > 0 else np.nan
    
    # Axis labels with bold font (using fontweight='bold')
    plt.xlabel("True RT (s)", fontsize=18, fontweight='bold')  # x-axis label
    plt.ylabel("Predicted RT (s)", fontsize=18, fontweight='bold')  # y-axis label
    
    # Add R² and MAE text to plot - 显示Outer CV的平均值±标准差
    plt.text(
        0.05, 0.95,                # Position: top-left (5%, 95%)
        f"R² = {r2_mean:.3f} ± {r2_std:.3f}\nMAE = {mae_mean:.2f} ± {mae_std:.2f}",  # Show 3 significant digits
        transform=ax.transAxes, 
        va='top',
        fontsize=16, 
        color=IPHONE_COLORS['text']  # Color: black (#000000)
    )
    
   
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_scatter.png", dpi=600)
    plt.close()

    # 残差图（保持原样，但更新样式）
    residuals = y_pred - y_true
    plt.figure(figsize=(6, 6))
    ax = plt.gca()
    
    # Apply iPhone-style axis settings
    iphone_style_ax(ax)
    
    plt.scatter(y_pred, residuals, alpha=0.8, s=70, color=IPHONE_COLORS['scatter'])
    plt.axhline(y=0, linestyle='--', color=IPHONE_COLORS['line'], linewidth=3)

    plt.xlabel("Predicted Retention Time (s)", fontsize=18, fontweight='bold')
    plt.ylabel("Residuals (Predicted - True)", fontsize=18, fontweight='bold')
    
    # 在残差图上也显示Outer CV的指标
    plt.text(
        0.05, 0.95,                # Position: top-left (5%, 95%)
        f"Outer CV R² = {r2_mean:.3f} ± {r2_std:.3f}\nOuter CV MAE = {mae_mean:.2f} ± {mae_std:.2f}",
        transform=ax.transAxes, 
        va='top',
        fontsize=16, 
        color=IPHONE_COLORS['text']
    )
    
    plt.tight_layout()
    plt.savefig(f"{save_prefix}_residuals.png", dpi=600)
    plt.close()

# ----------------- Optuna objective -----------------
def make_inner_objective(X_train, y_train, n_inner_splits=3):
    def objective(trial):
        C = trial.suggest_float('C', 1e-2, 1e3, log=True)
        gamma = trial.suggest_float('gamma', 1e-4, 1e1, log=True)
        epsilon = trial.suggest_float('epsilon', 1e-3, 1.0, log=True)

        model = SVR(C=C, gamma=gamma, epsilon=epsilon)
        kf_inner = KFold(n_splits=n_inner_splits, shuffle=True, random_state=SEED)

        inner_scores = []
        for tr_idx, val_idx in kf_inner.split(X_train):
            X_tr, X_val = X_train[tr_idx].copy(), X_train[val_idx].copy()
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            scaler = StandardScaler()
            X_tr[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_tr[:, :len(FEATURE_COLS)])
            X_val[:, :len(FEATURE_COLS)] = scaler.transform(X_val[:, :len(FEATURE_COLS)])

            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_val)
            inner_scores.append(r2_score(y_val, y_pred))

        return np.mean(inner_scores)
    return objective

# ----------------- nested CV -----------------
def nested_cv_evaluate(X, y, outer_splits=5, inner_splits=3, n_trials=100):
    kf_outer = KFold(n_splits=outer_splits, shuffle=True, random_state=SEED)
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(kf_outer.split(X), 1):
        print(f"\n--- Outer Fold {fold_idx}/{outer_splits} ---")
        X_train, X_val = X[train_idx].copy(), X[val_idx].copy()
        y_train, y_val = y[train_idx], y[val_idx]

        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=SEED))
        objective = make_inner_objective(X_train, y_train, n_inner_splits=inner_splits)
        study.optimize(objective, n_trials=n_trials, n_jobs=N_JOBS)
        best_params = study.best_params
        best_value = study.best_value
        print(f"  Inner best params: {best_params}, inner CV mean R2 = {best_value:.4f}")

        scaler = StandardScaler()
        X_train[:, :len(FEATURE_COLS)] = scaler.fit_transform(X_train[:, :len(FEATURE_COLS)])
        X_val[:, :len(FEATURE_COLS)] = scaler.transform(X_val[:, :len(FEATURE_COLS)])

        model = SVR(**best_params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)

        r2 = r2_score(y_val, y_pred)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        mae = mean_absolute_error(y_val, y_pred)

        print(f"  Outer fold {fold_idx} metrics - R2: {r2:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}")

        fold_results.append({
            'fold': fold_idx,
            'best_params': best_params,
            'inner_best_value': best_value,
            'r2': r2,
            'rmse': rmse,
            'mae': mae,
            'y_true': y_val,
            'y_pred': y_pred
        })

    r2s = [f['r2'] for f in fold_results]
    rmses = [f['rmse'] for f in fold_results]
    maes = [f['mae'] for f in fold_results]

    summary = {
        'r2_mean': np.mean(r2s),
        'r2_std': np.std(r2s, ddof=1),
        'rmse_mean': np.mean(rmses),
        'rmse_std': np.std(rmses, ddof=1),
        'mae_mean': np.mean(maes),
        'mae_std': np.std(maes, ddof=1)
    }

    return fold_results, summary

# ----------------- 单文件处理 -----------------
def process_single_file(df, file_prefix,
                        outer_splits=5, inner_splits=3, n_trials=100):
    print(f"\n{'='*50}")
    print(f"Processing {file_prefix}")
    print(f"{'='*50}")
    
    X = df[ALL_FEATURES].values
    y = df[TARGET_COL].values

    fold_results, summary = nested_cv_evaluate(X, y,
                                               outer_splits=outer_splits,
                                               inner_splits=inner_splits,
                                               n_trials=n_trials)

    print(f"\n{file_prefix} - Outer CV Summary (模型泛化性能):")
    print(f"  R²: {summary['r2_mean']:.4f} ± {summary['r2_std']:.4f}")
    print(f"  RMSE: {summary['rmse_mean']:.4f} ± {summary['rmse_std']:.4f}")
    print(f"  MAE: {summary['mae_mean']:.4f} ± {summary['mae_std']:.4f}")

    # 保存Outer CV的结果
    fold_preds = []
    for fr in fold_results:
        fold_df = pd.DataFrame({
            'y_true': fr['y_true'],
            'y_pred': fr['y_pred']
        })
        fold_df['fold'] = fr['fold']
        fold_preds.append(fold_df)
    all_fold_preds_df = pd.concat(fold_preds, ignore_index=True)
    all_fold_preds_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_nestedcv_outer_preds.csv"),
                             index=False)

    # 保存Outer CV的汇总指标
    summary_df = pd.DataFrame([{
        'file_prefix': file_prefix,
        'r2_mean': summary['r2_mean'],
        'r2_std': summary['r2_std'],
        'rmse_mean': summary['rmse_mean'],
        'rmse_std': summary['rmse_std'],
        'mae_mean': summary['mae_mean'],
        'mae_std': summary['mae_std']
    }])
    summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_nestedcv_summary.csv"), index=False)

    # 保存每个fold的最佳参数
    params_df = pd.DataFrame([fr['best_params'] for fr in fold_results])
    params_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_inner_best_params_per_fold.csv"), index=False)

    # 使用完整数据训练最终模型（用于学习曲线等）
    print("\nRunning final inner hyperparameter search on FULL data...")
    study_final = optuna.create_study(direction='maximize',
                                      sampler=optuna.samplers.TPESampler(seed=SEED))
    final_objective = make_inner_objective(X, y, n_inner_splits=inner_splits)
    study_final.optimize(final_objective, n_trials=n_trials, n_jobs=N_JOBS)
    final_best_params = study_final.best_params
    print(f"Final best params on FULL data: {final_best_params}")

    final_scaler = StandardScaler()
    X_scaled = X.copy()
    X_scaled[:, :len(FEATURE_COLS)] = final_scaler.fit_transform(X_scaled[:, :len(FEATURE_COLS)])
    final_model = SVR(**final_best_params)
    final_model.fit(X_scaled, y)

    # 保存最终模型和标准化器
    model_path = os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_final_svr_model.joblib")
    scaler_path = os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_final_scaler.joblib")
    joblib.dump(final_model, model_path)
    joblib.dump(final_scaler, scaler_path)

    pd.DataFrame([final_best_params]).to_csv(
        os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_final_best_params.csv"), index=False)

    # 绘制学习曲线
    lc_path = os.path.join(MODEL_SAVE_FOLDER, f"{file_prefix}_learning_curve.png")
    plot_learning_curve(final_model, X_scaled, y, f"Learning Curve - {file_prefix}", lc_path)

    # 绘制散点图和残差图 - 使用Outer CV的汇总结果
    y_true_all = all_fold_preds_df['y_true'].values
    y_pred_all = all_fold_preds_df['y_pred'].values
    plot_prefix = os.path.join(MODEL_SAVE_FOLDER, file_prefix)
    plot_scatter_and_residuals(y_true_all, y_pred_all, summary, plot_prefix)

    print(f"\nSaved final model to: {model_path}")
    print(f"Saved final scaler to: {scaler_path}")
    print(f"Saved plots with Outer CV metrics")
    print(f"{'='*50}\n")
    
    return summary

# ----------------- main -----------------
def main():
    print(f"{'='*50}")
    print("SVR Model Training for AM-IV, AM-V, AM-VI datasets")
    print(f"{'='*50}")
    
    data = load_data()
    
    # 检查加载的数据集
    loaded_prefixes = data['file_prefix'].unique()
    print(f"Loaded datasets: {list(loaded_prefixes)}")
    print(f"Total samples: {len(data)}")
    
    summaries = []
    for file_prefix in ALLOWED_PREFIXES:
        if file_prefix in data['file_prefix'].unique():
            df_group = data[data['file_prefix'] == file_prefix].copy()
            s = process_single_file(df_group, file_prefix,
                                    outer_splits=5, inner_splits=3, n_trials=100)
            summaries.append({'file_prefix': file_prefix, **s})
        else:
            print(f"\nWarning: {file_prefix} not found in data, skipping.")
    
    # 保存所有数据集的汇总结果
    if summaries:
        all_summary_df = pd.DataFrame(summaries)
        all_summary_df.to_csv(os.path.join(MODEL_SAVE_FOLDER, "all_files_nestedcv_summary.csv"), index=False)
        
        print("\n" + "="*50)
        print("FINAL RESULTS - Outer CV Performance Summary:")
        print("="*50)
        for idx, row in all_summary_df.iterrows():
            print(f"\n{row['file_prefix']}:")
            print(f"  R²: {row['r2_mean']:.4f} ± {row['r2_std']:.4f}")
            print(f"  RMSE: {row['rmse_mean']:.4f} ± {row['rmse_std']:.4f}")
            print(f"  MAE: {row['mae_mean']:.4f} ± {row['mae_std']:.4f}")
        print("="*50)
    else:
        print("\nNo datasets were successfully processed.")

if __name__ == "__main__":
    main()

SVR Model Training for AM-IV, AM-V, AM-VI datasets


[I 2026-02-28 13:20:48,298] A new study created in memory with name: no-name-7b9025c1-0676-48e1-abf4-868ebc971550


Loaded datasets: ['AM-III-filtered', 'AM-IV-filtered', 'AM-V-filtered', 'AM-VI-filtered']
Total samples: 1264

Processing AM-IV-filtered

--- Outer Fold 1/5 ---


[I 2026-02-28 13:20:48,640] Trial 1 finished with value: -0.038708677973818904 and parameters: {'C': 0.03298406160279723, 'gamma': 0.0002559658178690078, 'epsilon': 0.08621845910311986}. Best is trial 1 with value: -0.038708677973818904.
[I 2026-02-28 13:20:48,725] Trial 2 finished with value: 0.7184073071066003 and parameters: {'C': 124.01120650078855, 'gamma': 0.0006755298587590949, 'epsilon': 0.12669672440181537}. Best is trial 2 with value: 0.7184073071066003.
[I 2026-02-28 13:20:48,787] Trial 3 finished with value: -0.042083818547121364 and parameters: {'C': 0.33440184258850114, 'gamma': 1.681140615572798, 'epsilon': 0.2311922585474392}. Best is trial 2 with value: 0.7184073071066003.
[I 2026-02-28 13:20:48,795] Trial 7 finished with value: -0.047410357609624455 and parameters: {'C': 0.04132515217787251, 'gamma': 1.5401176181590428, 'epsilon': 0.5081098643097511}. Best is trial 2 with value: 0.7184073071066003.
[I 2026-02-28 13:20:48,802] Trial 5 finished with value: 0.74443285758

  Inner best params: {'C': 172.86687114143132, 'gamma': 0.005646990873017703, 'epsilon': 0.003724811362350794}, inner CV mean R2 = 0.8007
  Outer fold 1 metrics - R2: 0.8934, RMSE: 3.7088, MAE: 2.6998

--- Outer Fold 2/5 ---


...

  Inner best params: {'C': 979.489283856447, 'gamma': 0.0010352703990188768, 'epsilon': 0.18208523076292776}, inner CV mean R2 = 0.9173


[I 2026-02-28 13:23:05,815] A new study created in memory with name: no-name-399c809d-5fb0-4be0-b3ab-cfe3bd2ec908


  Outer fold 5 metrics - R2: 0.9691, RMSE: 2.1625, MAE: 1.4113

AM-III-filtered - Outer CV Summary (模型泛化性能):
  R²: 0.9407 ± 0.0327
  RMSE: 3.0517 ± 0.9884
  MAE: 1.6457 ± 0.3220

Running final inner hyperparameter search on FULL data...


[I 2026-02-28 13:23:06,604] Trial 1 finished with value: -0.03186169672123351 and parameters: {'C': 1.1375849384664438, 'gamma': 4.35803892814732, 'epsilon': 0.07502416743729784}. Best is trial 1 with value: -0.03186169672123351.
[I 2026-02-28 13:23:06,645] Trial 5 finished with value: -0.006369027585418907 and parameters: {'C': 0.08912668528117687, 'gamma': 0.0026509971224625223, 'epsilon': 0.00362057846200797}. Best is trial 5 with value: -0.006369027585418907.
[I 2026-02-28 13:23:06,655] Trial 2 finished with value: 0.4254915893448606 and parameters: {'C': 4.104272866834851, 'gamma': 0.0013896407867506667, 'epsilon': 0.0016473717505303283}. Best is trial 2 with value: 0.4254915893448606.
[I 2026-02-28 13:23:06,667] Trial 3 finished with value: -0.0348707429955919 and parameters: {'C': 0.0421759676654567, 'gamma': 2.361922496973699, 'epsilon': 0.003791559880345765}. Best is trial 2 with value: 0.4254915893448606.
[I 2026-02-28 13:23:06,669] Trial 21 finished with value: 0.90949908913

Final best params on FULL data: {'C': 158.34086190206887, 'gamma': 0.005246983178203215, 'epsilon': 0.00783197552777361}

Saved final model to: ./2-svr-model-other4/AM-III-filtered_final_svr_model.joblib
Saved final scaler to: ./2-svr-model-other4/AM-III-filtered_final_scaler.joblib
Saved plots with Outer CV metrics


FINAL RESULTS - Outer CV Performance Summary:

AM-IV-filtered:
  R²: 0.8425 ± 0.0531
  RMSE: 3.9894 ± 0.2581
  MAE: 2.7759 ± 0.1047

AM-VI-filtered:
  R²: 0.9185 ± 0.0568
  RMSE: 4.9050 ± 1.2624
  MAE: 2.7533 ± 0.2058

AM-V-filtered:
  R²: 0.8484 ± 0.0258
  RMSE: 2.7993 ± 0.2856
  MAE: 2.1165 ± 0.2153

AM-III-filtered:
  R²: 0.9407 ± 0.0327
  RMSE: 3.0517 ± 0.9884
  MAE: 1.6457 ± 0.3220
